# JadeDragon — End-to-end demo on one page

Two paths:
  * **Path A (no training)** — uses LGRC2024 ground-truth pitch labels + a hand-edited lyric JSON.
    Run this first to verify the alignment + MusicXML + audio path works.
  * **Path B (trained model)** — uses the YOLO checkpoint from `01_train_pitch_detector.ipynb`
    plus PaddleOCR for lyrics. The actual end-to-end MVP.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/JadeDragon/JadeDragon Transcriber'
%cd {PROJECT}

In [ ]:
# Install deps. PaddleOCR + paddlepaddle are heavy and only needed for Path B.
!pip install -q ultralytics music21 Pillow
!apt-get -qq install fluidsynth
# For Path B (lyric OCR):
# !pip install -q paddleocr paddlepaddle

In [ ]:
# Download a free GM SoundFont for fluidsynth (FluidR3 GM, ~140 MB)
import os, urllib.request
SF2 = '/content/FluidR3_GM.sf2'
if not os.path.exists(SF2):
    url = 'https://github.com/musescore/MuseScore/raw/master/share/sound/FluidR3Mono_GM.sf3'
    # Note: .sf3 is also accepted by fluidsynth >= 2.0
    SF2 = '/content/FluidR3Mono_GM.sf3'
    if not os.path.exists(SF2):
        urllib.request.urlretrieve(url, SF2)
print('SoundFont:', SF2)

## Path A — no training needed

Uses ground-truth labels for one page + a manual lyric JSON. The fastest way to
verify the pipeline produces something audible.

In [ ]:
# 1. Bootstrap a placeholder lyric JSON for page 100
!python scripts/bootstrap_lyrics.py --page 100 --overlay
# Inspect the preview to see where it placed lyric boxes:
from PIL import Image
Image.open('demo_lyrics/100.preview.png').resize((600, 850))

In [ ]:
# 2. Edit demo_lyrics/100.json: replace each '？' with the correct lyric character
#    (read it off the page image). Then run:
!python scripts/demo_one_page.py --page 100 --wav --soundfont {SF2}

In [ ]:
# 3. Listen to the rendered audio
from IPython.display import Audio, display
display(Audio('outputs/demo/100.wav'))

## Path B — trained model + automatic lyric OCR

In [ ]:
# Install paddle stack (heavy, ~2 minutes)
!pip install -q paddleocr paddlepaddle

In [ ]:
from jade_dragon.pipeline import transcribe_with_detector
import os

weights = f'{PROJECT}/checkpoints/lgrc2024_yolov8m_best.pt'
image = f'{PROJECT}/LGRC2024 dataset/datasets/images/val/100.jpg'

result = transcribe_with_detector(
    image_path=image,
    weights=weights,
    out_dir='outputs/path_b',
    use_paddle_ocr=True,
    render_wav=True,
    soundfont_path=SF2,
)
for k, v in result.items():
    if k == 'aligned': continue
    print(f'{k}: {v}')

In [ ]:
from IPython.display import Audio
Audio(result['wav'])